# Whisker-Pole Contact Classifier — Inference

Run the trained EfficientNet-B3 on a video and produce:
1. **Per-frame CSV** — every frame with its contact prediction (0/1) and probability
2. **Interval CSV** — contiguous contact regions in `Start,End` format

Batch
1, 3, 5, 15, 16

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

INFERENCE_DIR = os.path.dirname(os.path.abspath("__file__"))
if INFERENCE_DIR not in sys.path:
    sys.path.insert(0, INFERENCE_DIR)

from utils import (
    load_model,
    get_inference_transforms,
    VideoFrameDataset,
    run_inference,
    frames_to_intervals,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1 — Configuration

In [ ]:
# ========================  PATHS  ========================

VIDEO_PATH = "/home/mvdokh/data/102525_1.mp4"


CHECKPOINT_PATH = os.path.join(
    os.path.dirname(INFERENCE_DIR),
    "Training", "checkpoints", "contact_classifier.pt"
)

# Output CSVs will be saved here
OUTPUT_DIR = os.path.join(INFERENCE_DIR, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ========================  SETTINGS  ========================

IMG_SIZE      = 256
BATCH_SIZE    = 64       # can be larger than training since no gradients
NUM_WORKERS   = 2        # try higher to parallelize video decoding
THRESHOLD     = 0.5      # probability threshold for contact
START_FRAME   = 0
END_FRAME     = 250_000  # only first 250k frames (second half has no pole)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

## 2 — Load Model

In [ ]:
model = load_model(CHECKPOINT_PATH, DEVICE)

## 3 — Create Video Dataset & DataLoader

In [ ]:
transform = get_inference_transforms(IMG_SIZE)

dataset = VideoFrameDataset(
    video_path=VIDEO_PATH,
    start_frame=START_FRAME,
    end_frame=END_FRAME,
    transform=transform,
)

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Total frames to process: {len(dataset):,}")
print(f"Batches: {len(dataloader):,}")

## 4 — Run Inference

In [ ]:
results_df = run_inference(model, dataloader, DEVICE, threshold=THRESHOLD)

n_contact = (results_df["contact"] == 1).sum()
n_no_contact = (results_df["contact"] == 0).sum()
print(f"\nTotal frames   : {len(results_df):,}")
print(f"Contact frames : {n_contact:,} ({100*n_contact/len(results_df):.1f}%)")
print(f"No contact     : {n_no_contact:,} ({100*n_no_contact/len(results_df):.1f}%)")
print()
results_df.head(10)

## 5 — Convert to Intervals & Save CSVs

In [ ]:
# Per-frame CSV
per_frame_path = os.path.join(OUTPUT_DIR, "per_frame_predictions.csv")
results_df.to_csv(per_frame_path, index=False)
print(f"Saved per-frame predictions → {per_frame_path}")

# Contact intervals CSV
intervals_df = frames_to_intervals(results_df, label_col="contact", label_val=1)
intervals_path = os.path.join(OUTPUT_DIR, "contact_intervals.csv")
intervals_df.to_csv(intervals_path, index=False)
print(f"Saved contact intervals     → {intervals_path}")
print(f"\nFound {len(intervals_df)} contact intervals")
print()
intervals_df.head(20)

## 6 — Quick Summary

In [ ]:
# Interval length stats
intervals_df["length"] = intervals_df["End"] - intervals_df["Start"] + 1

print(f"Contact intervals: {len(intervals_df)}")
print(f"Total contact frames: {intervals_df['length'].sum():,}")
print(f"Avg interval length: {intervals_df['length'].mean():.1f} frames")
print(f"Min interval length: {intervals_df['length'].min()} frames")
print(f"Max interval length: {intervals_df['length'].max()} frames")
print(f"Median interval length: {intervals_df['length'].median():.0f} frames")

## 7 — Batch Inference over `/home/mvdokh/data`

Run the same contact classifier over **all videos** under a root folder and write a `contact_intervals.csv` (or `<video_name>_contact_intervals.csv` if a folder has multiple videos) into each folder that contains videos.


In [ ]:
# Root directory containing folders with videos
DATA_ROOT = "/home/mvdokh/data"

print(f"Batch data root: {DATA_ROOT}")

In [ ]:
from utils import batch_contact_intervals

# Use the same settings as above unless you want to override them here
batch_outputs = batch_contact_intervals(
    model,
    DEVICE,
    DATA_ROOT,
    start_frame=START_FRAME,
    end_frame=END_FRAME,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    threshold=THRESHOLD,
)

print("\nBatch inference complete. Wrote interval CSVs to:")
for folder, csvs in batch_outputs.items():
    print(f"  {folder}")
    for csv in csvs:
        print(f"    - {csv}")